# Layerwise Irrep Stage Visualization (Sample 2)

This notebook traces the latest Ti_Al masked A1 SR model stage-by-stage, plots all irrep channels spatially,
and decodes each stage back to IPF for comparison against LR/HR targets.

- Sample index: `2`
- Decoder settings: **normal config settings** (no overrides)


In [1]:
import sys
import math
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt


def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'training').is_dir() and (p / 'models').is_dir() and (p / 'experiments').is_dir():
            return p
    raise FileNotFoundError(
        "Could not find repo root containing training/, models/, and experiments/. "
        f"Started from: {start}"
    )


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from training.config_utils import load_and_prepare_config
from training.quaternion_dataset import QuaternionDataset
from utils.runtime_helpers import build_iso_embedding_sr_attn_from_config, load_checkpoint_state_compat
from utils.symmetry_utils import resolve_symmetry
from visualization.ipf_render import render_ipf_rgb

torch.set_grad_enabled(True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


def to_hwc_quat(q: torch.Tensor) -> torch.Tensor:
    if q.dim() != 3:
        raise ValueError(f'Expected rank-3 tensor, got {tuple(q.shape)}')
    if q.shape[-1] == 4:
        return q
    if q.shape[0] == 4:
        return q.permute(1, 2, 0)
    raise ValueError(f'Could not find quaternion channel axis in shape {tuple(q.shape)}')


def normalize_quat(q: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    q = q / torch.linalg.norm(q, dim=-1, keepdim=True).clamp_min(eps)
    q = torch.where(q[..., :1] < 0.0, -q, q)
    return q


def quat_ang_err_deg(pred: torch.Tensor, gt: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    pred_n = normalize_quat(pred)
    gt_n = normalize_quat(gt)
    dots = torch.sum(pred_n * gt_n, dim=-1).abs().clamp(max=1.0 - eps)
    return 2.0 * torch.rad2deg(torch.acos(dots))


Using device: cuda


In [2]:
# Paths / controls
DATASET_ROOT = Path('/data/warren/materials/EBSD/Ti_Al_1pct_QSR_x4')
EXP_CONFIG = REPO_ROOT / 'experiments/Ti_Al_1pct/iso_embedding_sr_attn_a1_masked_hcp_01/config.json'
CHECKPOINT = REPO_ROOT / 'experiments/Ti_Al_1pct/iso_embedding_sr_attn_a1_masked_hcp_01/checkpoints/best_model.pt'

SPLIT = 'Test'
SAMPLE_INDEX = 2
IPF_REF_DIR = 'Z'
CHANNELS_PER_ROW = 6
DECODE_EACH_STAGE = True
STAGE_FILTER = None  # e.g. ['encode_a1_lr', 'upsample_conv', 'conv_hr1']

assert DATASET_ROOT.exists(), f'Missing dataset_root: {DATASET_ROOT}'
assert EXP_CONFIG.exists(), f'Missing config: {EXP_CONFIG}'
assert CHECKPOINT.exists(), f'Missing checkpoint: {CHECKPOINT}'


In [ ]:
# Load config/model/checkpoint with normal decoder settings from config
cfg = load_and_prepare_config(EXP_CONFIG)
cfg.dataset_root = str(DATASET_ROOT)

model = build_iso_embedding_sr_attn_from_config(cfg, device=device)
ckpt = torch.load(CHECKPOINT, map_location=device)
state_dict = ckpt.get('model_state_dict', ckpt)
load_checkpoint_state_compat(model, state_dict, context='layerwise notebook checkpoint')
model.eval()

if hasattr(model, 'decoder'):
    backend = str(getattr(cfg, 'decoder_backend', 'optimizing')).lower()
    if backend != 'optimizing':
        raise ValueError(f"Expected decoder_backend='optimizing', got {backend!r}")

    cfg_resolution = int(getattr(cfg, 'decoder_cubochoric_resolution', 1))
    cfg_num_starts = int(getattr(cfg, 'decoder_num_starts', int(model.decoder.num_starts)))
    cfg_steps = int(getattr(cfg, 'decoder_steps', int(model.decoder.steps)))
    cfg_lr = float(getattr(cfg, 'decoder_lr', float(model.decoder.lr)))

    model_resolution = int(getattr(model.decoder, 'cubochoric_resolution', cfg_resolution))
    if model_resolution != cfg_resolution:
        raise RuntimeError(
            f'Decoder cubochoric resolution mismatch: model={model_resolution}, config={cfg_resolution}'
        )

    # Normal settings (config-driven)
    model.decoder.num_starts = cfg_num_starts
    model.decoder.steps = cfg_steps
    model.decoder.lr = cfg_lr

    print('Decoder backend: optimizing')
    print('Active normal decoder settings:')
    print(f'  cubochoric_resolution = {int(model.decoder.cubochoric_resolution)}')
    print(f'  num_starts            = {int(model.decoder.num_starts)}')
    print(f'  steps                 = {int(model.decoder.steps)}')
    print(f'  lr                    = {float(model.decoder.lr)}')

print(f'Model class: {model.__class__.__module__}.{model.__class__.__name__}')


Using symmetry group from config: D6h
Overridden config values:
  - dataset_root: /data/warren/materials/EBSD/Ti_Al_1pct_QSR_x4 (default: )
  - epochs: 50 (default: 10)
  - batch_size: 2 (default: 4)
  - symmetry_group: D6h (default: O)
  - crystal: hcp (default: fcc)
  - use_attention: False (default: True)
  - decoder_num_starts: 8 (default: 2)
  - decoder_steps: 12 (default: 1)
  - decoder_lr: 0.03 (default: 0.05)
  - decoder_table_cache_dir: /home/warren/projects/Reynolds-QSR/out/decoder_lookup_tables (default: out/decoder_lookup_tables)
  - num_workers: 8 (default: 0)
  - preload: True (default: False)
  - preload_torch: True (default: False)
  - min_free_cuda_gb: 4.0 (default: 0.0)
  - model.type: iso_embedding_sr_attn_a1_masked (default: iso_embedding_sr_attn)
Using existing symmetry files for D6h


In [ ]:
# Load sample 2
sym_class = resolve_symmetry(getattr(cfg, 'symmetry_group', 'D6h'))
ds = QuaternionDataset(dataset_root=str(DATASET_ROOT), split=SPLIT, preload=False, preload_torch=False)

lr_raw, hr_raw = ds[SAMPLE_INDEX]
lr_hwc = normalize_quat(to_hwc_quat(lr_raw).to(device=device, dtype=torch.float32))
hr_hwc = normalize_quat(to_hwc_quat(hr_raw).to(device=device, dtype=torch.float32))

h_lr, w_lr = int(lr_hwc.shape[0]), int(lr_hwc.shape[1])
h_hr, w_hr = int(hr_hwc.shape[0]), int(hr_hwc.shape[1])

print(f'Loaded sample {SAMPLE_INDEX} from split={SPLIT}')
print(f'LR shape: {tuple(lr_hwc.shape)}')
print(f'HR shape: {tuple(hr_hwc.shape)}')


In [ ]:
def _sanitize_for_viz(x: torch.Tensor) -> torch.Tensor:
    return torch.nan_to_num(x, nan=0.0, posinf=1e4, neginf=-1e4).clamp(-1e4, 1e4)


def _build_irrep_blocks(irreps) -> list[dict]:
    blocks: list[dict] = []
    start = 0
    for mul, ir in irreps:
        mul_i = int(mul)
        dim_i = int(ir.dim)
        n = mul_i * dim_i
        blocks.append({
            'start': start,
            'mul': mul_i,
            'dim': dim_i,
            'L': int(ir.l),
            'ir': str(ir),
            'end': start + n,
        })
        start += n
    return blocks


def _resize_quat_target(target_hwc: torch.Tensor, size_hw: tuple[int, int]) -> torch.Tensor:
    h, w = int(target_hwc.shape[0]), int(target_hwc.shape[1])
    if (h, w) == size_hw:
        return target_hwc
    q = target_hwc.permute(2, 0, 1).unsqueeze(0)
    q = F.interpolate(q, size=size_hw, mode='bilinear', align_corners=False).squeeze(0).permute(1, 2, 0)
    return normalize_quat(q)


def _decode_feature_map_to_quat(model_obj, feat_flat: torch.Tensor, hw: tuple[int, int]) -> torch.Tensor:
    H, W = hw
    with torch.enable_grad():
        q_flat = model_obj.decode(feat_flat)
    return normalize_quat(q_flat.reshape(H, W, 4))


def _apply_attention_with_trace(model_obj, feat_flat: torch.Tensor, hr_shape: tuple[int, int]):
    traces = []
    if (not model_obj.use_attention) or (len(model_obj.attention_blocks) == 0):
        return feat_flat, traces

    Hr, Wr = hr_shape
    feat = feat_flat.unsqueeze(0)
    B, N, C = feat.shape
    if N != Hr * Wr:
        raise ValueError(f'Attention trace expected N={Hr*Wr}, got {N}')

    block_h = min(int(model_obj.hr_attn_block_size), Hr)
    block_w = min(int(model_obj.hr_attn_block_size), Wr)
    pad_h = (-Hr) % block_h
    pad_w = (-Wr) % block_w
    Hr_pad, Wr_pad = Hr + pad_h, Wr + pad_w

    feat_pad = feat
    if pad_h > 0 or pad_w > 0:
        f2d = feat_pad.reshape(B, Hr, Wr, C).permute(0, 3, 1, 2)
        f2d = F.pad(f2d, (0, pad_w, 0, pad_h), mode='reflect')
        feat_pad = f2d.permute(0, 2, 3, 1).reshape(B, Hr_pad * Wr_pad, C)

    d_block = model_obj._get_hr_sh_block(block_h, block_w, feat_pad.device, feat_pad.dtype)

    for bi, block in enumerate(model_obj.attention_blocks, start=1):
        delta = block(feat_pad, d_block, Hr_pad, Wr_pad, block_h, block_w)
        feat_pad = _sanitize_for_viz(feat_pad + delta)

        feat_unpad = feat_pad
        if pad_h > 0 or pad_w > 0:
            feat_unpad = feat_pad.reshape(B, Hr_pad, Wr_pad, C)[:, :Hr, :Wr, :].reshape(B, Hr * Wr, C)

        traces.append({
            'name': f'attn_block_{bi}',
            'feat': feat_unpad.squeeze(0).detach().clone(),
            'shape': (Hr, Wr),
        })

    feat_final = feat_pad
    if pad_h > 0 or pad_w > 0:
        feat_final = feat_final.reshape(B, Hr_pad, Wr_pad, C)[:, :Hr, :Wr, :].reshape(B, Hr * Wr, C)

    return feat_final.squeeze(0), traces


@torch.no_grad()
def trace_model_sr_stages(model_obj, lr_hwc_in: torch.Tensor):
    lr_hwc_in = normalize_quat(lr_hwc_in)
    Hlr, Wlr = int(lr_hwc_in.shape[0]), int(lr_hwc_in.shape[1])
    lr_shape = (Hlr, Wlr)

    feats = []

    feat = model_obj.encode_a1(lr_hwc_in.reshape(Hlr * Wlr, 4))
    feat = _sanitize_for_viz(feat)
    feats.append({'name': 'encode_a1_lr', 'feat': feat.detach().clone(), 'shape': lr_shape})

    if bool(model_obj.use_lr_conv1):
        feat = model_obj.conv_lr1(feat, lr_shape)
        feat = _sanitize_for_viz(feat)
        feats.append({'name': 'conv_lr1', 'feat': feat.detach().clone(), 'shape': lr_shape})

    if bool(model_obj.use_lr_conv2):
        feat = model_obj.conv_lr2(feat, lr_shape)
        feat = _sanitize_for_viz(feat)
        feats.append({'name': 'conv_lr2', 'feat': feat.detach().clone(), 'shape': lr_shape})

    feat, hr_shape = model_obj.upsample_conv(feat, lr_shape)
    feat = _sanitize_for_viz(feat)
    feats.append({'name': 'upsample_conv', 'feat': feat.detach().clone(), 'shape': hr_shape})

    feat = model_obj.conv_hr1(feat, hr_shape)
    feat = _sanitize_for_viz(feat)
    feats.append({'name': 'conv_hr1', 'feat': feat.detach().clone(), 'shape': hr_shape})

    feat, attn_traces = _apply_attention_with_trace(model_obj, feat, hr_shape)
    feat = _sanitize_for_viz(feat)
    feats.extend(attn_traces)

    if getattr(model_obj, 'boundary_snap', None) is not None:
        feat = model_obj.boundary_snap(feat, hr_shape)
        feat = _sanitize_for_viz(feat)
        feats.append({'name': 'boundary_snap', 'feat': feat.detach().clone(), 'shape': hr_shape})

    feats.append({'name': 'final_feat_before_decode', 'feat': feat.detach().clone(), 'shape': hr_shape})
    return feats


def plot_stage_decoded_top(
    stage_name: str,
    q_pred_hwc: torch.Tensor,
    q_tgt_hwc: torch.Tensor,
    sym_class_obj,
    ref_dir: str,
):
    err = quat_ang_err_deg(q_pred_hwc, q_tgt_hwc)
    err_np = err.detach().cpu().numpy()

    rgb_pred = render_ipf_rgb(q_pred_hwc.detach().cpu().numpy().astype(np.float32), sym_class_obj, ref_dir=ref_dir)
    rgb_tgt = render_ipf_rgb(q_tgt_hwc.detach().cpu().numpy().astype(np.float32), sym_class_obj, ref_dir=ref_dir)

    vmax = float(np.percentile(err_np, 99.0))
    vmax = max(vmax, 1e-3)

    fig, axes = plt.subplots(1, 3, figsize=(13, 4.0), constrained_layout=True)
    axes[0].imshow(rgb_pred)
    axes[0].set_title(f'{stage_name} decoded IPF-{ref_dir}')
    axes[0].axis('off')

    axes[1].imshow(rgb_tgt)
    axes[1].set_title(f'Target IPF-{ref_dir}')
    axes[1].axis('off')

    im = axes[2].imshow(err_np, cmap='magma', vmin=0.0, vmax=vmax)
    axes[2].set_title('Angular error (deg)')
    axes[2].axis('off')
    fig.colorbar(im, ax=axes[2], shrink=0.85)
    plt.show()


def plot_irrep_group_with_row_colorbars(feat_np: np.ndarray, block: dict, group_title: str):
    """
    Plot one figure for an irrep group.
    - One row per copy.
    - One channel map per m component in that row.
    - One shared colorbar for the whole row.
    """
    start = int(block['start'])
    mul = int(block['mul'])
    dim = int(block['dim'])
    L = int(block['L'])

    m_vals = list(range(-L, L + 1))
    if len(m_vals) != dim:
        # Fallback for non-(2L+1) shapes (should not happen for O(3) irreps).
        m_vals = list(range(dim))

    fig = plt.figure(figsize=(2.9 * (dim + 1), 3.0 * mul), constrained_layout=True)
    gs = fig.add_gridspec(
        nrows=mul,
        ncols=dim + 1,
        width_ratios=[1.0] * dim + [0.08],
    )

    for copy_idx in range(mul):
        row_maps = []
        row_channels = []
        for j in range(dim):
            ch = start + copy_idx * dim + j
            row_channels.append(ch)
            row_maps.append(feat_np[..., ch])

        row_stack = np.stack(row_maps, axis=0)
        vmax = float(np.percentile(np.abs(row_stack), 99.0))
        vmax = max(vmax, 1e-8)

        im_last = None
        for j, ch in enumerate(row_channels):
            ax = fig.add_subplot(gs[copy_idx, j])
            m = m_vals[j]
            title = f"L={L}, m={m:+d}, copy {copy_idx + 1} [ch {ch}]"
            im_last = ax.imshow(feat_np[..., ch], cmap='coolwarm', vmin=-vmax, vmax=vmax)
            ax.set_title(title, fontsize=8)
            ax.axis('off')

        cax = fig.add_subplot(gs[copy_idx, -1])
        cbar = fig.colorbar(im_last, cax=cax)
        cbar.ax.tick_params(labelsize=7)
        cbar.set_label('Feature value', fontsize=8)

    fig.suptitle(group_title, fontsize=13)
    plt.show()


In [ ]:
# Run layerwise trace + plots (decoded shown at top of each stage)
irrep_blocks = _build_irrep_blocks(model.irreps_a1)
stage_trace = trace_model_sr_stages(model, lr_hwc)

print('Captured stages:')
for i, st in enumerate(stage_trace):
    h, w = st['shape']
    c = int(st['feat'].shape[-1])
    print(f'  [{i:02d}] {st["name"]:<24s} -> shape=({h},{w},{c})')

for st in stage_trace:
    name = st['name']
    if (STAGE_FILTER is not None) and (name not in set(STAGE_FILTER)):
        continue

    feat_flat = st['feat']
    hw = st['shape']
    H, W = hw
    feat_np = feat_flat.reshape(H, W, -1).detach().cpu().numpy()

    if DECODE_EACH_STAGE:
        q_pred = _decode_feature_map_to_quat(model, feat_flat, hw)

        if hw == (int(lr_hwc.shape[0]), int(lr_hwc.shape[1])):
            q_tgt = lr_hwc
        elif hw == (int(hr_hwc.shape[0]), int(hr_hwc.shape[1])):
            q_tgt = hr_hwc
        else:
            q_tgt = _resize_quat_target(hr_hwc, hw)

        # Decoded map shown first (top)
        plot_stage_decoded_top(name, q_pred, q_tgt, sym_class, ref_dir=IPF_REF_DIR)

    # Then irrep-group channel maps with one colorbar per copy-row
    for gi, block in enumerate(irrep_blocks):
        group_title = (
            f"{name} | g{gi}: {block['mul']}x{block['ir']} "
            f"channels[{block['start']}:{block['end']}]"
        )
        plot_irrep_group_with_row_colorbars(feat_np, block, group_title)
